# A Dose-Response Evaluation of Explanation Metrics under Controlled Shortcut Reliance in CNN Image Classifiers

## Amendment A2, session 2: final SD2 CIFAR-10 ResNet-18 doses and the SD1 confirmatory set

Run only after the two dated A2 completion notes (CIFAR pilot r_c → dose set;
SD1 pilot r_c → dose set) are committed in `docs/PREDICTIONS_amendments.md`.

1. Fill `CIFAR_DOSES` and `SD1_DOSES` in the parameters cell from those notes.
2. **+ Add Input**: the bundle dataset and the Output of the session-1 notebook
   (it holds `a2_session1_results.zip`, so the CIFAR pilot checkpoint is reused).
3. **Settings → Accelerator → GPU T4 x2**, **Internet on**, then
   **Save Version → Save & Run All (Commit)**.

Each non-empty list is trained with seeds 42 1 2 and scored; the results are
packed into `a2_session2_results.zip` in the notebook's Output.

In [ ]:
# Parameters: copy the final doses from the dated A2 completion notes
# (docs/PREDICTIONS_amendments.md), e.g. CIFAR_DOSES = [-0.09, -0.075, -0.06, -0.05].
CIFAR_DOSES = []  # cifar10_resnet18, seeds 42 1 2
SD1_DOSES = []    # pet_resnet18_pretrained, seeds 42 1 2
SEEDS = [42, 1, 2]
assert CIFAR_DOSES or SD1_DOSES, 'Fill CIFAR_DOSES and/or SD1_DOSES from the A2 completion notes first.'

In [ ]:
# 1. Copy the uploaded project into the writable working folder
import glob, os, shutil, subprocess, sys
found = glob.glob('/kaggle/input/**/dose_response.py', recursive=True)
assert found, 'Project files not found: add your uploaded dataset with "+ Add Input".'
src = os.path.dirname(found[0])
WORK = '/kaggle/working/project'
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
print('project copied from', src)
print(sorted(os.listdir(WORK)))

# Reuse the session-1 checkpoints (train skips models whose metrics JSON exists)
import zipfile
for archive in glob.glob('/kaggle/input/**/a2_session1_results.zip', recursive=True):
    with zipfile.ZipFile(archive) as z:
        z.extractall(WORK)
    print('extracted', archive)

In [ ]:
# 2. Install the one missing package, check the GPU and the data, get the ImageNet weights
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'captum==0.9.0'], check=True)
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU: turn on Settings → Accelerator → GPU, then restart.'
for name in ('trainval.npz', 'test.npz'):
    assert os.path.exists(f'data/pet128/{name}'), f'data/pet128/{name} missing: rebuild the bundle with -Dataset a2.'
assert os.path.isdir('data/cifar-10-batches-py'), 'data/cifar-10-batches-py missing: rebuild the bundle with -Dataset a2.'
print('Pet and CIFAR-10 data found')
if SD1_DOSES:
    from torchvision.models import ResNet18_Weights, resnet18
    try:
        resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  # cached for the training cells
    except Exception as error:
        raise RuntimeError('ImageNet weights download failed: turn Internet on (Settings → Internet).') from error
    print('ImageNet ResNet-18 weights ready')

In [ ]:
# 3. Train and score every model of the filled lists
def run(*args):
    subprocess.run([sys.executable, '-u', '-W', 'ignore', 'dose_response.py', *args], check=True)

JOBS = [(base, doses) for base, doses in (('cifar10_resnet18', CIFAR_DOSES), ('pet_resnet18_pretrained', SD1_DOSES))
        if doses]
for base, doses in JOBS:
    selection = ['--bases', base, '--shortcuts', 'patch', '--doses', *map(str, doses),
                 '--seeds', *map(str, SEEDS)]
    run('train', *selection)
    run('evaluate', *selection)
    run('placebo', *selection)
    run('behaviour', '--selection-only', *selection)

In [ ]:
# 4. Pack the results to download (only the models of this session's lists)
sys.path.insert(0, WORK)
import dose_response
names = [dose_response.model_name(base, 'patch', dose, seed) for base, doses in JOBS for dose in doses for seed in SEEDS]
files = sorted({f for n in names for f in glob.glob(f'checkpoint/dose/{n}.pt') + glob.glob(f'checkpoint/dose/{n}_metrics.json')
                + glob.glob(f'results/dose_response/per_model/{n}_*.csv')})
out = '/kaggle/working/a2_session2_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f)  # keeps checkpoint/... and results/... paths
print(len(files), 'files for', len(names), 'models ->', out)
shutil.rmtree(WORK)  # keep only the zip in the Output